# ek_freq_eda
FFT-based period detection · decomposition · anomaly overlay

In [2]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d
from sentinel.ml_logic.data import find_anomaly_segments
import json
from pathlib import Path

In [ ]:
# ── config ───────────────────────────────────────────────────────────────────
CHANNELS = [
    'channel_41', 'channel_42', 'channel_43',
    'channel_44', 'channel_45', 'channel_46',
]
DECOMP_CH       = CHANNELS[0]   # channel shown in 4-panel decomposition
N_PEAKS         = 2             # dominant periods to find per channel
DS_FACTOR       = 10            # downsample factor for FFT
MAX_PERIOD      = 3_000_000     # cap to avoid red-noise false peak at Nyquist edge
ZOOM_LEN        = 15_000        # rows in short-scale zoom
ZOOM_START_FRAC = 0.35          # position of zoom (fraction of nominal rows)

# ── decomposition windows (uniform_filter1d passes periods >> W) ─────────────
BIG_WINDOW    = 50_000   # >> big_sin period → captures orbital trend
MEDIUM_WINDOW = 500      # >> small_period (≈98) but << BIG_WINDOW → captures medium osc
SMALL_WINDOW  = 30       # ≈ small_period / 3 → captures fast oscillation

ANOMALY_COLOR = '#e74c3c'
NOMINAL_COLOR = '#2980b9'

RAW_DIR = Path('../data/raw')
OUT_DIR = Path('../data')

In [4]:
cols  = ['id', 'is_anomaly'] + CHANNELS
train = pq.read_table(str(RAW_DIR / 'train.parquet'), columns=cols
                      ).to_pandas().set_index('id')
print(f'Loaded {len(train):,} rows')
print(f'Anomaly rate: {train["is_anomaly"].mean():.4%}')

nom = train[train['is_anomaly'] == 0]
print(f'Nominal rows: {len(nom):,}')

Loaded 14,728,321 rows
Anomaly rate: 10.4839%
Nominal rows: 13,184,217


In [ ]:
def find_periods(series, n_peaks=2, min_period=50, max_period=None, ds_factor=10):
    """FFT on a downsampled version of series. Returns list of {period, power}."""
    x = series.values[::ds_factor].astype(np.float64)
    x -= x.mean()
    N     = len(x)
    freqs = np.fft.rfftfreq(N)                  # 0 … 0.5 cycles/sample
    power = np.abs(np.fft.rfft(x)) ** 2

    min_freq = 2.0 / N                           # need ≥ 2 full cycles visible
    if max_period is not None:
        min_freq = max(min_freq, ds_factor / max_period)
    max_freq = 1.0 / (min_period / ds_factor)
    search   = np.where((freqs > min_freq) & (freqs <= max_freq), power, 0.0)

    found = []
    for _ in range(n_peaks):
        idx = int(np.argmax(search))
        if search[idx] == 0:
            break
        period_orig = int(round(ds_factor / freqs[idx]))
        found.append({'period': period_orig, 'power': float(power[idx])})
        w = max(3, idx // 8)
        search[max(0, idx - w): idx + w + 1] = 0.0

    return sorted(found, key=lambda d: d['period'])

In [ ]:
freq_map = {}
for ch in CHANNELS:
    peaks = find_periods(nom[ch], n_peaks=N_PEAKS, ds_factor=DS_FACTOR,
                         max_period=MAX_PERIOD)
    freq_map[ch] = [p['period'] for p in peaks]
    print(f'{ch}: {freq_map[ch]}')

with open(OUT_DIR / 'freq_map.json', 'w') as f:
    json.dump(freq_map, f, indent=2)
print('\nSaved freq_map.json')

In [ ]:
def decompose(arr, big_window, medium_window, small_window):
    """Three-level cascade decomposition.

    sin1: slow orbital trend  (passes periods >> big_window)
    sin2: medium oscillation  (passes periods >> medium_window, << big_window)
    sin3: fast oscillation    (passes periods >> small_window, << medium_window)
    res:  true residual       (anomaly signal + measurement noise)
    """
    a    = np.asarray(arr, dtype=np.float64)
    sin1 = uniform_filter1d(a,  size=big_window,    mode='nearest')
    r1   = a    - sin1
    sin2 = uniform_filter1d(r1, size=medium_window, mode='nearest')
    r2   = r1   - sin2
    sin3 = uniform_filter1d(r2, size=small_window,  mode='nearest')
    res  = r2   - sin3
    return sin1, sin2, sin3, res

## Viz 1 — Full-length: raw + big sinusoid + anomaly regions

In [ ]:
segs = find_anomaly_segments(train['is_anomaly'].values)

fig, axes = plt.subplots(len(CHANNELS), 1,
                         figsize=(18, 2.5 * len(CHANNELS)), sharex=True)
axes = np.atleast_1d(axes)

for ax, ch in zip(axes, CHANNELS):
    raw     = train[ch].values
    big_sin = uniform_filter1d(raw.astype(np.float64), size=BIG_WINDOW, mode='nearest')
    ax.plot(raw,     lw=0.15, alpha=0.5, color=NOMINAL_COLOR, label='raw')
    ax.plot(big_sin, lw=1.5,  color='tomato', label=f'big_sin  w={BIG_WINDOW:,}')
    for seg in segs:
        ax.axvspan(seg['start'], seg['end'],
                   color=ANOMALY_COLOR, alpha=0.35, linewidth=0)
    ax.set_ylabel(ch, fontsize=8)
    ax.legend(fontsize=7, loc='upper right')

axes[0].set_title(
    'Full length · raw + big sinusoid overlay · anomaly regions shaded', fontsize=11)
axes[-1].set_xlabel('Row index')
plt.tight_layout()
plt.savefig(OUT_DIR / 'freq_eda_full.png', dpi=72, bbox_inches='tight')
plt.show()

## Viz 2 — Zoom: big + small sinusoid both visible

In [ ]:
nom_idx    = np.where(train['is_anomaly'].values == 0)[0]
zoom_start = nom_idx[int(len(nom_idx) * ZOOM_START_FRAC)]
zoom_end   = zoom_start + ZOOM_LEN

fig, axes = plt.subplots(len(CHANNELS), 1,
                         figsize=(15, 2.5 * len(CHANNELS)), sharex=True)
axes = np.atleast_1d(axes)

for ax, ch in zip(axes, CHANNELS):
    raw_seg = train[ch].values[zoom_start:zoom_end]
    sp      = (freq_map.get(ch) or [2000])[0]
    trend   = uniform_filter1d(raw_seg.astype(np.float64),
                               size=sp * 3, mode='nearest')
    xs      = np.arange(zoom_start, zoom_end)
    ax.plot(xs, raw_seg, lw=0.4, alpha=0.8, color=NOMINAL_COLOR, label='raw')
    ax.plot(xs, trend,   lw=2,   color='tomato',
            label=f'trend  w={sp*3:,}')
    ax.set_ylabel(ch, fontsize=8)
    ax.legend(fontsize=7, loc='upper right')

axes[0].set_title(
    f'Zoom rows {zoom_start:,}–{zoom_end:,}  (big + small sinusoid visible)', fontsize=11)
axes[-1].set_xlabel('Row index')
plt.tight_layout()
plt.savefig(OUT_DIR / 'freq_eda_zoom.png', dpi=100, bbox_inches='tight')
plt.show()

## Viz 3 — Power spectrum per channel

In [ ]:
fig, axes = plt.subplots(len(CHANNELS), 2,
                         figsize=(16, 2.8 * len(CHANNELS)))
axes = axes.reshape(len(CHANNELS), 2)

for i, ch in enumerate(CHANNELS):
    x     = nom[ch].values[::DS_FACTOR].astype(np.float64)
    x    -= x.mean()
    freqs = np.fft.rfftfreq(len(x))
    power = np.abs(np.fft.rfft(x)) ** 2
    per_orig = DS_FACTOR / freqs[1:]
    pw       = power[1:]

    ax_l, ax_r = axes[i]

    # left: full log-log spectrum (shows 1/f slope and all peaks)
    ax_l.loglog(per_orig, pw, lw=0.5, color=NOMINAL_COLOR)
    ax_l.set_xlabel('Period (samples, log scale)')
    ax_l.set_ylabel('Power')
    ax_l.set_title(f'{ch}')

    # right: zoom on short periods to show the small sinusoid peak clearly
    mask = per_orig < 500
    ax_r.semilogy(per_orig[mask], pw[mask], lw=0.8, color=NOMINAL_COLOR)
    ax_r.set_xlabel('Period (samples)')
    ax_r.set_title(f'{ch} — periods < 500')

    for p in freq_map.get(ch, []):
        for ax in (ax_l, ax_r):
            ax.axvline(p, color='tomato', ls='--', lw=1.2, label=f'T={p:,}')
    axes[i, 0].legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR / 'freq_eda_spectrum.png', dpi=100, bbox_inches='tight')
plt.show()

## Viz 4 — Signal decomposition (one channel)

In [ ]:
ch = DECOMP_CH
nom_start = int(len(nom) * 0.3)
seg_raw   = nom[ch].values[nom_start: nom_start + ZOOM_LEN].astype(np.float64)

# derive windows from detected periods so decomposition is data-driven
sp = (freq_map.get(ch) or [98])[0]   # smallest detected period (e.g. 98)
bw = min(BIG_WINDOW, len(seg_raw) // 4)   # BIG_WINDOW is >> any medium period
mw = max(10, sp * 5)                       # passes medium osc, kills fast (≈ 5× sp)
sw = max(3,  sp // 3)                      # extracts fast oscillation (≈ sp/3)

print(f'Decomposition windows — big: {bw:,}  medium: {mw}  small: {sw}  (sp={sp})')

sin1, sin2, sin3, res = decompose(seg_raw, bw, mw, sw)
r1 = seg_raw - sin1
r2 = r1      - sin2
r3 = r2      - sin3   # same as res

panels = [
    (seg_raw, sin1, 'a)  raw  +  sin1  (big orbital trend)',        NOMINAL_COLOR,  'tomato'),
    (r1,      sin2, 'b)  raw−sin1  +  sin2  (medium oscillation)', NOMINAL_COLOR,  'seagreen'),
    (r2,      sin3, 'c)  raw−sin1−sin2  +  sin3  (fast osc)',      NOMINAL_COLOR,  'goldenrod'),
    (r3,      None, 'd)  residual = raw−sin1−sin2−sin3',            'mediumpurple', None),
]

fig, axes = plt.subplots(4, 1, figsize=(15, 11), sharex=True)
for ax, (base, comp, title, c_base, c_comp) in zip(axes, panels):
    ax.plot(base, lw=0.5, alpha=0.7, color=c_base)
    if comp is not None:
        ax.plot(comp, lw=1.8, color=c_comp)
    ax.set_title(title, fontsize=10)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel(f'Row offset from nominal position {nom_start:,}')
fig.suptitle(
    f'Cascaded decomposition: {ch}  '
    f'(bw={bw:,}  mw={mw}  sw={sw}  sp={sp})',
    fontsize=12)
plt.tight_layout()
plt.savefig(OUT_DIR / 'freq_eda_decomp.png', dpi=100, bbox_inches='tight')
plt.show()